In [3]:
!pip install scikit-learn --quiet

In [ ]:
student_answer = """au) Su) oen ee ae eee eee ee maa\nCoperation flood] were set ve by the government, bot this led to\n| development 10. concentrated areas, 2\na 3) Farmers were given crop insyrapedaabinst failure of crersi?\n( case of ‘droughts, fleeds. Fires cic. Establisnment of grameen\nj anks , cooperatives and banks that provicted loans at reasonable\n( totes of \\nterest: poten abies aearh\n| 3) Kisan Credit Cork, PALS ( Personal _acctdbet Insurance Scheme},\n| fenumerotive prices, Special weethec bulletins Special. programmes\n1 on Tvand radio channels were also eee eee ee we\n__|€xpleHation_by mictelemen and specolaters.
"""

print(f"Word count: {len(student_answer.split())}")

Word count: 99


In [ ]:
TEACHER_ANSWER_FILE = "teacher_answer.txt"   # <-- change if needed

with open(TEACHER_ANSWER_FILE, "r", encoding="utf-8") as f:
    teacher_answer = f.read().strip()

print(f"Word count: {len(teacher_answer.split())}")
print("\nPreview:")
print(teacher_answer[:300], "..." if len(teacher_answer) > 300 else "")

Word count: 42

Preview:
# Option A — single answer (plain text)
Green revolution and white revolution were set up by the government.
Farmers were given crop insurance against failure of crops in case of droughts.
Kisan Credit Card and PAIS were introduced to reduce exploitation. 


In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def clean(text):
    """Remove OCR noise characters, normalize whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def compute_similarity(student_text, teacher_text):
    """Returns cosine similarity between two texts using TF-IDF."""
    s = clean(student_text)
    t = clean(teacher_text)

    vectorizer = TfidfVectorizer()
    tfidf      = vectorizer.fit_transform([s, t])
    sim        = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
    return round(float(sim), 4)


def similarity_to_marks(sim, total=10):
    """Scale similarity (0–1) to marks out of total."""
    if sim >= 0.85: marks = 9.0 + (sim - 0.85) / 0.15
    elif sim >= 0.70: marks = 7.0 + (sim - 0.70) / 0.15 * 2
    elif sim >= 0.55: marks = 5.0 + (sim - 0.55) / 0.15 * 2
    elif sim >= 0.40: marks = 3.0 + (sim - 0.40) / 0.15 * 2
    else: marks = max(0.0, sim / 0.40 * 3)
    return round(min(marks, total), 1)


def grade(marks, total=10):
    r = marks / total
    if r >= 0.85: return "Excellent"
    if r >= 0.70: return "Good"
    if r >= 0.55: return "Average"
    if r >= 0.40: return "Below Average"
    return "Poor"


similarity = compute_similarity(student_answer, teacher_answer)
marks      = similarity_to_marks(similarity)
grade_str  = grade(marks)

print("Scoring done")

Scoring done


In [ ]:
sep = "═" * 55
print(sep)
print("MARKING REPORT")
print(sep)
print(f"  Similarity Score : {similarity:.4f}  (out of 1.0)")
print(f"  Marks Awarded    : {marks} / 10")
print(f"  Grade            : {grade_str}")
print(sep)

from sklearn.feature_extraction.text import CountVectorizer

student_words = set(clean(student_answer).split())
teacher_words = set(clean(teacher_answer).split())

stopwords = {'and','the','of','in','to','a','is','are','was','were',
             'by','on','at','for','with','this','that','it','be','an',
             'also','but','or','not','from','as','up','set','also'}
student_kw = student_words - stopwords
teacher_kw = teacher_words - stopwords

matched   = student_kw & teacher_kw
missed    = teacher_kw - student_kw

print(f"\n  Keywords matched : {len(matched)} → {', '.join(sorted(matched)[:15])}")
print(f"  Keywords missed  : {len(missed)}  → {', '.join(sorted(missed)[:15])}")
print(sep)

═══════════════════════════════════════════════════════
MARKING REPORT
═══════════════════════════════════════════════════════
  Similarity Score : 0.2959  (out of 1.0)
  Marks Awarded    : 2.2 / 10
  Grade            : Poor
═══════════════════════════════════════════════════════

  Keywords matched : 10 → case, credit, crop, droughts, failure, farmers, given, government, insurance, kisan
  Keywords missed  : 15  → against, answer, card, crops, exploitation, green, introduced, option, pais, plain, reduce, revolution, single, text, white
═══════════════════════════════════════════════════════
